# Lab 1 – From data to a first classifier: decision trees on *Bank Marketing*

**Machine Learning** – Master in Data Science and Business Analytics – A.Y. 2026/27
Lab tutor: Francesco Casadei

### Business context
A Portuguese bank runs telephone marketing campaigns to sell a **term deposit**. Every row of the dataset is a contact with a client; the target `y` says whether the client subscribed.

> *Business question*: **before calling**, can we predict which clients are likely to subscribe, so that the call centre focuses its effort on them?

**Dataset**: [Bank Marketing – UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing), file `bank-additional-full.csv` (41,188 contacts, 20 attributes + target). Source: Moro S., Cortez P., Rita P. (2014), *A data-driven approach to predict the success of bank telemarketing*, Decision Support Systems 62: 22–31.

### Goals of the lab
1. explore the data quickly, looking for what matters for modelling
2. prepare the data: hidden missing values, data leakage, encoding of categorical attributes
3. split the data with the **holdout** method and set a **baseline**
4. train, visualise and interpret a **decision tree**
5. observe **overfitting** when the tree grows

Indicative timing: EDA 20' · preparation 25' · holdout and baseline 15' · first tree 30' · overfitting 30'

## 0. Setup
Put `bank-additional-full.csv` in the same folder as this notebook (on Google Colab: upload it with the *Files* panel on the left).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42          # fix the randomness: everybody gets the same results
DATA_PATH = 'bank-additional-full.csv'
sns.set_theme(style='whitegrid')

## 1. Load the data
Open the raw file with a text viewer (Jupyter, VS Code and Colab can all show it) and check:
- the **separator** character
- whether the first row is a **header**
- how strings are quoted

Then read it with [`pd.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) into a dataframe called `df` and show its shape and first rows.

In [ ]:
# TODO: read the file; which separator do you need?
df = pd.read_csv(DATA_PATH, ...)
print(df.shape)
df.head()

### Attributes
| Group | Attributes |
|---|---|
| Client | `age`, `job`, `marital`, `education`, `default` (has credit in default?), `housing` (housing loan?), `loan` (personal loan?) |
| Last contact of the current campaign | `contact` (cellular/telephone), `month`, `day_of_week`, `duration` (duration of the call, in seconds) |
| Campaign history | `campaign` (number of contacts in this campaign), `pdays` (days since the client was last contacted in a previous campaign; `999` = never contacted), `previous` (contacts before this campaign), `poutcome` (outcome of the previous campaign) |
| Socio-economic context | `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed` (quarterly/monthly indicators) |
| **Target** | `y`: has the client subscribed a term deposit? (`yes`/`no`) |

In [ ]:
df.info()

**Question**: according to `info()`, are there missing values? Keep the answer in mind for section 2.3.

## 2. Exploratory analysis (compact)
The complete EDA checklist was discussed in session 2: here we focus on what matters for building a classifier.

### 2.1 The target
Show the absolute and relative frequencies of `y` (`value_counts`, option `normalize=True`) and a bar chart (`sns.countplot`).

In [ ]:
# TODO: absolute and relative frequencies of the target, and a countplot
...

**Question**: a trivial "model" that always answers `no` – what accuracy would it get? Is the dataset balanced?

### 2.2 Numeric attributes
Show a statistical summary of the numeric attributes (`describe`, transposed for readability) and their histograms.

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
# TODO: transposed summary of the numeric attributes
...

In [ ]:
# TODO: histograms of the numeric attributes (try bins=30 and a larger figsize)
...

**Questions**
- Look at `pdays`: median, quartiles, maximum. What is happening?
- Which distributions are strongly skewed?
- The socio-economic attributes have very few distinct values: why? (hint: what do they describe?)

### 2.3 Categorical attributes and *hidden* missing values
`info()` reports no missing values, but several categorical attributes contain the string `'unknown'`. Count, for each categorical attribute, how many rows are `'unknown'`.

In [ ]:
cat_cols = [c for c in df.columns if c not in num_cols and c != 'y']
print(cat_cols)
# TODO: number of 'unknown' values per categorical attribute, sorted in descending order
...

In [ ]:
# TODO: frequencies of the values of 'default'
...

**Question**: what would you do with `'unknown'`? Drop the rows? Replace it with the most frequent value? Keep it as a category of its own? Look at `default` in particular.

### 2.4 Duplicates

In [ ]:
# TODO: how many fully duplicated rows?
...

### 2.5 Attributes vs target
A quick way to see if a categorical attribute is related to the target: the **subscription rate** (fraction of `yes`) for each of its values. The function below draws it; the dashed line is the overall rate.

In [ ]:
base_rate = (df['y'] == 'yes').mean()

def plot_rate(col, ax):
    rate = (df['y'] == 'yes').groupby(df[col]).mean().sort_values()
    rate.plot.barh(ax=ax)
    ax.axvline(base_rate, ls='--', c='k')
    ax.set_title(f'Subscription rate by {col}')
    ax.set_xlabel('rate of yes')

# TODO: choose three categorical attributes and draw their subscription rates side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, [..., ..., ...]):
    plot_rate(col, ax)
plt.tight_layout()

For the numeric attributes, compare the distributions for `yes` and `no` with boxplots (`x='y'`).

In [ ]:
cols = ['age', 'campaign', 'duration', 'euribor3m', 'nr.employed']
fig, axes = plt.subplots(1, len(cols), figsize=(20, 4))
for ax, c in zip(axes, cols):
    # TODO: boxplot of c for each value of the target
    ...
plt.tight_layout()

**Question**: which attribute seems to separate `yes` and `no` best? Keep it in mind: we will come back to it immediately.

## 3. Data preparation

### 3.1 Duplicates
Remove the duplicated rows.

In [ ]:
# TODO: remove the duplicates and check the new shape
...

### 3.2 Data leakage: the case of `duration`
Think about **when** each attribute is known.

**Question**: the bank wants to use the model to decide *whom to call*. Is `duration` available at that moment? What would happen if we kept it?

In [ ]:
df_leak = df.copy()                       # kept only for the optional section 7
# TODO: drop 'duration' from df
df = ...

### 3.3 The code 999 in `pdays`
For a decision tree the value 999 is not a big problem: a split such as `pdays <= 998` separates "previously contacted" from "never contacted". For other methods (e.g. *k*-nearest neighbours, Lab 4) 999 would be treated as a huge distance. We make the information explicit with a binary attribute `prev_contacted`.

In [ ]:
# TODO: create 'prev_contacted' = 1 if the client was contacted in a previous campaign, 0 otherwise
df['prev_contacted'] = ...
df['prev_contacted'].value_counts()

### 3.4 Predictors and target
scikit-learn wants a matrix of predictors `X` and a vector `y`. Encode the target as `1` = `yes`, `0` = `no`.

In [ ]:
# TODO: target as 0/1 and predictors without the target
y = ...
X = ...
print(X.shape, y.mean().round(3))

### 3.5 Encoding the categorical attributes
scikit-learn trees accept only numbers. With **one-hot encoding** each categorical attribute with *k* values becomes *k* binary columns. Use [`pd.get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) with `dtype=int`.

*Note*: here encoding before the split is harmless, because it does not use statistics computed on the data. In Lab 2 we will do it the clean way, with `OneHotEncoder` inside a scikit-learn `Pipeline`.

In [ ]:
# TODO: one-hot encode the categorical attributes listed in cat_cols
X = ...
print(X.shape)
X.columns.tolist()[:25]

## 4. Holdout and baseline

### 4.1 Train/test split
Split the data into 70% training and 30% test with [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), using `stratify=y` and `random_state=RANDOM_STATE`. Check the rate of `yes` in the two parts.

In [ ]:
# TODO: stratified 70/30 holdout
X_train, X_test, y_train, y_test = train_test_split(...)
print(X_train.shape, X_test.shape)
print(f'rate of yes - train: {y_train.mean():.3f}  test: {y_test.mean():.3f}')

**Question**: why `stratify=y`? What could happen without it, with a minority class this small?

### 4.2 Baseline
A [`DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) with `strategy='most_frequent'` always predicts the majority class. Train it and compute its test accuracy: this is the number to beat.

In [ ]:
# TODO: train a DummyClassifier (most_frequent) and compute its accuracy on the test set
dummy = ...
...
baseline_acc = ...
print(f'Baseline test accuracy: {baseline_acc:.3f}')

## 5. A first decision tree
Train a [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) with `max_depth=3`. Use `criterion='entropy'`, i.e. splits chosen by **information gain** as seen in class (the scikit-learn default is `'gini'`, which usually gives very similar trees).

Compute the accuracy on the **training** set and on the **test** set.

In [ ]:
# TODO: tree with max_depth=3 and criterion='entropy'
tree3 = ...
...

acc_train = ...
acc_test = ...
print(f'train accuracy: {acc_train:.4f}   test accuracy: {acc_test:.4f}   (baseline {baseline_acc:.4f})')

### 5.1 Visualise the tree
Use [`plot_tree`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html) with the names of the features and of the classes. In each node: the split condition, the entropy, the number of training samples, the class distribution `value = [no, yes]` and the majority class.

In [ ]:
plt.figure(figsize=(24, 9))
# TODO: plot tree3 with feature_names, class_names=['no', 'yes'], filled=True
...
plt.show()

The same tree as text, convenient to read the rules:

In [ ]:
print(export_text(tree3, feature_names=list(X.columns), show_weights=True))

**Questions**
1. Which attribute is used at the root? Why do you think it was chosen?
2. Find the leaf with the highest proportion of `yes`. Translate the path from the root into a business rule ("clients who … ").
3. How many leaves predict `yes`? Compare the test accuracy with the baseline: how much did we gain?

### 5.2 Which attributes does the tree use?
`feature_importances_` measures how much each attribute contributes to the reduction of impurity in the tree.

In [ ]:
# TODO: bar chart of the non-zero feature importances of tree3
imp = pd.Series(tree3.feature_importances_, index=X.columns)
...

## 6. Overfitting

### 6.1 A fully grown tree
Train a tree **without** limits on the depth. How deep is it? How many leaves (`get_depth()`, `get_n_leaves()`)? Compare training and test accuracy.

In [ ]:
# TODO: tree without depth limit; print depth, number of leaves, train and test accuracy
full_tree = ...
...

### 6.2 Accuracy as a function of the depth
For `max_depth` from 1 to 30, train a tree and store training and test accuracy; then plot the two curves together with the baseline.

In [ ]:
depths = range(1, 31)
train_acc, test_acc = [], []
for d in depths:
    # TODO: train a tree with max_depth=d and append the two accuracies
    ...

plt.figure(figsize=(9, 5))
# TODO: plot train_acc and test_acc vs depths, plus the baseline as a horizontal line
...
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.legend()
plt.title('Decision tree: accuracy vs depth');

**Questions**
1. Why does the training accuracy keep growing towards 1?
2. Where is the test accuracy highest? What happens beyond that point, and why?
3. We have just chosen the "best" depth by looking at the **test** set. Is the corresponding test accuracy a fair estimate of the performance on new clients?

### 6.3 Exercise: limiting the size of the leaves
Overfitting can also be controlled with `min_samples_leaf` (minimum number of training samples in a leaf). Repeat the experiment of 6.2 without limiting the depth, for `min_samples_leaf` in `[1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]`. Use a logarithmic *x* axis (`plt.xscale('log')`). Which value works best? How large is the tree (number of leaves) at that value?

In [ ]:
leaf_sizes = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
# TODO: same experiment as 6.2, also storing the number of leaves of each tree
...

## 7. Optional: what if we had kept `duration`?
Repeat the preparation on `df_leak` (which still contains `duration`), using **the same rows** for training and test, and train a tree of depth 5 with and without `duration`. Compare the test accuracies and also the **share of the real subscribers that each model identifies** (among the test clients with `y = 1`, the fraction predicted as `1`).

In [ ]:
df_leak['prev_contacted'] = (df_leak['pdays'] != 999).astype(int)
X_leak = pd.get_dummies(df_leak.drop(columns=['y']), columns=cat_cols, dtype=int)

# same split as before, selected through the row index
X_leak_train, X_leak_test = X_leak.loc[X_train.index], X_leak.loc[X_test.index]

# TODO: train a depth-5 tree on (X_train, X_test) and on (X_leak_train, X_leak_test);
#       for each, print the test accuracy and the share of real subscribers identified: pred[y_test == 1].mean()
...

**Question**: the model with `duration` is clearly "better". Why would it be a serious mistake to deliver it to the bank?

## 8. Wrap-up
Discuss with your neighbour and write two lines for each question:
1. The best tree gains only a little accuracy over the baseline. Does this mean it is useless for the bank? What would you rather measure? → *Lab 3: performance measures*
2. We chose `max_depth` looking at the test set. How should we choose it, keeping the test set apart? → *Lab 2: validation and cross-validation*
3. Change `RANDOM_STATE` and re-run sections 4–6. Does the best depth change? What does this say about a single holdout split? → *Lab 2*